# ClipNova sur Google Colab

Ce carnet lance l'application ClipNova complète (sous-titres automatiques, meilleurs moments, export MP4)
sur un ordinateur gratuit de Google. Tu l'utilises ensuite depuis ton iPad ou ton téléphone.

**Mode d'emploi :** touche **Exécution → Tout exécuter** (ou ▶ sur chaque cellule, dans l'ordre).
Attends 2 à 3 minutes. À la fin, un lien `https://….trycloudflare.com` s'affiche : ouvre-le.

**Garde cet onglet Colab ouvert** pendant que tu utilises ClipNova. Si tu le fermes, l'application s'arrête.

In [ ]:
# 1) Installation (2 minutes environ)
!pip -q install fastapi uvicorn python-multipart yt-dlp faster-whisper "opencv-python-headless>=4.9,<5"
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print("✅ Installation terminée")

In [ ]:
# 2) Fichiers de l'application
import base64, pathlib
FILES = {"clipnova.py": "...", "server.py": "IiIiU2VydmV1ciB3ZWIgZGUgQ2xpcE5vdmEgOiBvdXZyZSBodHRwOi8vPGlwLWR1LXBjPjo4MDAwIGRlcHVpcyB0b24gdMOpbMOpcGhvbmUuIiIiCmltcG9ydCBqc29uCmltcG9ydCBpbwppbXBvcnQgc2h1dGlsCmltcG9ydCB6aXBmaWxlCgpmcm9tIGZhc3RhcGkgaW1wb3J0IEZhc3RBUEksIEZpbGUsIEZvcm0sIEhUVFBFeGNlcHRpb24sIFVwbG9hZEZpbGUKZnJvbSBmYXN0YXBpLnJlc3BvbnNlcyBpbXBvcnQgRmlsZVJlc3BvbnNlLCBTdHJlYW1pbmdSZXNwb25zZQpmcm9tIGZhc3RhcGkuc3RhdGljZmlsZXMgaW1wb3J0IFN0YXRpY0ZpbGVzCgppbXBvcnQgY2xpcG5vdmEKCmFwcCA9IEZhc3RBUEkodGl0bGU9IkNsaXBOb3ZhIikKYXBwLm1vdW50KCIvZmlsZXMiLCBTdGF0aWNGaWxlcyhkaXJlY3Rvcnk9Y2xpcG5vdmEuV09SSyksIG5hbWU9ImZpbGVzIikKYXBwLm1vdW50KCIvc3RhdGljIiwgU3RhdGljRmlsZXMoZGlyZWN0b3J5PWNsaXBub3ZhLkJBU0UgLyAic3RhdGljIiksIG5hbWU9InN0YXRpYyIpCgoKQGFwcC5nZXQoIi8iKQpkZWYgaG9tZSgpOgogICAgcmV0dXJuIEZpbGVSZXNwb25zZShjbGlwbm92YS5CQVNFIC8gInN0YXRpYyIgLyAiaW5kZXguaHRtbCIpCgoKQGFwcC5nZXQoIi9jbGlwcyIpCmRlZiBjbGlwc19wYWdlKCk6CiAgICByZXR1cm4gRmlsZVJlc3BvbnNlKGNsaXBub3ZhLkJBU0UgLyAic3RhdGljIiAvICJjbGlwcy5odG1sIikKCgpAYXBwLnBvc3QoIi9hcGkvam9icyIpCmRlZiBjcmVhdGVfam9iKHVybDogc3RyID0gRm9ybSgiIiksIGZpbGU6IFVwbG9hZEZpbGUgfCBOb25lID0gRmlsZShOb25lKSwgbGFuZzogc3RyID0gRm9ybSgiZnIiKSwKICAgICAgICAgICAgICAgbWluX2Q6IGZsb2F0ID0gRm9ybSgyMCksIG1heF9kOiBmbG9hdCA9IEZvcm0oNjApLCBjb3VudDogaW50ID0gRm9ybSgxMCksCiAgICAgICAgICAgICAgIG1vZGVsOiBzdHIgPSBGb3JtKCJzbWFsbCIpLCBjb2xvcjogc3RyID0gRm9ybSgiRkZFNTAwIiksIHN1YnM6IGJvb2wgPSBGb3JtKFRydWUpLAogICAgICAgICAgICAgICBhaTogYm9vbCA9IEZvcm0oVHJ1ZSksIGtleXdvcmRzOiBzdHIgPSBGb3JtKCIiKSwgdF9mcm9tOiBmbG9hdCA9IEZvcm0oMCksIHRfdG86IGZsb2F0ID0gRm9ybSgwKSwKICAgICAgICAgICAgICAgcmlnaHRzOiBib29sID0gRm9ybShGYWxzZSkpOgogICAgaWYgbm90IHVybC5zdHJpcCgpIGFuZCBub3QgKGZpbGUgYW5kIGZpbGUuZmlsZW5hbWUpOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oNDAwLCAiRG9ubmUgdW4gbGllbiBvdSB1bmUgdmlkw6lvLiIpCiAgICBpZiBub3QgcmlnaHRzOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oNDAwLCAiQ29jaGUgwqsgSmUgcG9zc8OoZGUgbGVzIGRyb2l0cyBkZSBjZXR0ZSB2aWTDqW8gwrsgcG91ciBjb250aW51ZXIuIikKICAgIGlmIG1pbl9kID49IG1heF9kOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oNDAwLCAiTGEgZHVyw6llIG1pbiBkb2l0IMOqdHJlIHBsdXMgcGV0aXRlIHF1ZSBsYSBkdXLDqWUgbWF4LiIpCiAgICBqb2JfaWQsIGZvbGRlciA9IGNsaXBub3ZhLm5ld19qb2IoKQogICAgdXBsb2FkID0gTm9uZQogICAgaWYgZmlsZSBhbmQgZmlsZS5maWxlbmFtZToKICAgICAgICB1cGxvYWQgPSBmb2xkZXIgLyAoInNvdXJjZSIgKyAoY2xpcG5vdmEuUGF0aChmaWxlLmZpbGVuYW1lKS5zdWZmaXggb3IgIi5tcDQiKSkKICAgICAgICB3aXRoIHVwbG9hZC5vcGVuKCJ3YiIpIGFzIGY6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmaWxlLmZpbGUsIGYpCiAgICBvcHRzID0geyJsYW5nIjogbGFuZyBpZiBsYW5nICE9ICJhdXRvIiBlbHNlIE5vbmUsICJtaW4iOiBtaW5fZCwgIm1heCI6IG1heF9kLAogICAgICAgICAgICAiY291bnQiOiBtYXgoMSwgbWluKGNvdW50LCAzMCkpLCAibW9kZWwiOiBtb2RlbCwgImNvbG9yIjogY29sb3IsICJzdWJzIjogc3VicywKICAgICAgICAgICAgImFpIjogYWksICJrZXl3b3JkcyI6IGtleXdvcmRzLCAidF9mcm9tIjogdF9mcm9tLCAidF90byI6IHRfdG99CiAgICBjbGlwbm92YS5KT0JTW2pvYl9pZF1bIm5hbWUiXSA9IChmaWxlLmZpbGVuYW1lIGlmIHVwbG9hZCBlbHNlIHVybC5zdHJpcCgpKVs6MTIwXQogICAgY2xpcG5vdmEuc3RhcnRfam9iKGpvYl9pZCwgTm9uZSBpZiB1cGxvYWQgZWxzZSB1cmwuc3RyaXAoKSwgdXBsb2FkLCBvcHRzKQogICAgcmV0dXJuIHsiaWQiOiBqb2JfaWR9CgoKQGFwcC5nZXQoIi9hcGkvaW5mbyIpCmRlZiB2aWRlb19pbmZvKHVybDogc3RyKToKICAgICIiIlRpdHJlLCBkdXLDqWUgZXQgbWluaWF0dXJlIGQndW4gbGllbiwgcG91ciBsYSBwYWdlIGRlIHLDqWdsYWdlcy4iIiIKICAgIHRyeToKICAgICAgICBvdXQgPSBjbGlwbm92YS5ydW4oWyJ5dC1kbHAiLCAiLS1za2lwLWRvd25sb2FkIiwgIi0tbm8td2FybmluZ3MiLCAiLS1wcmludCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiJSh0aXRsZSlzXG4lKGR1cmF0aW9uKXNcbiUodGh1bWJuYWlsKXMiLCB1cmxdKS5zcGxpdGxpbmVzKCkKICAgIGV4Y2VwdCBSdW50aW1lRXJyb3IgYXMgZToKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKDQwMCwgIkxpZW4gaWxsaXNpYmxlIDogdmlkw6lvIHByaXbDqWUsIHN1cHByaW3DqWUgb3Ugc2l0ZSBub24gcHJpcyBlbiBjaGFyZ2UuIikgZnJvbSBlCiAgICBkdXIgPSBmbG9hdChvdXRbMV0pIGlmIGxlbihvdXQpID4gMSBhbmQgb3V0WzFdIG5vdCBpbiAoIiIsICJOQSIpIGVsc2UgMAogICAgcmV0dXJuIHsidGl0bGUiOiBvdXRbMF0gaWYgb3V0IGVsc2UgIiIsICJkdXJhdGlvbiI6IGR1ciwgInRodW1ibmFpbCI6IG91dFsyXSBpZiBsZW4ob3V0KSA+IDIgZWxzZSAiIn0KCgpAYXBwLmdldCgiL2FwaS9wcm9qZWN0cyIpCmRlZiBwcm9qZWN0cygpOgogICAgcmV0dXJuIHNvcnRlZCgoeyJpZCI6IGosICoqe2s6IHYgZm9yIGssIHYgaW4gY2xpcG5vdmEuc25hcHNob3QoaikuaXRlbXMoKSBpZiBrICE9ICJjbGlwcyJ9LAogICAgICAgICAgICAgICAgICAgICJjb3VudCI6IGxlbihjbGlwbm92YS5KT0JTW2pdWyJyYXciXSksCiAgICAgICAgICAgICAgICAgICAgInRodW1iIjogY2xpcG5vdmEuc25hcHNob3QoailbImNsaXBzIl1bMF1bImZpbGUiXSBpZiBjbGlwbm92YS5KT0JTW2pdWyJyYXciXSBlbHNlICIifQogICAgICAgICAgICAgICAgICAgZm9yIGogaW4gY2xpcG5vdmEuSk9CUyksIGtleT1sYW1iZGEgeDogLXhbImNyZWF0ZWQiXSkKCgpkZWYgZ2V0X2pvYihqb2JfaWQ6IHN0cikgLT4gZGljdDoKICAgIGlmIGpvYl9pZCBub3QgaW4gY2xpcG5vdmEuSk9CUzoKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKDQwNCwgIlTDomNoZSBpbmNvbm51ZSAobGUgc2VydmV1ciBhIHBldXQtw6p0cmUgcmVkw6ltYXJyw6kpLiIpCiAgICByZXR1cm4gY2xpcG5vdmEuSk9CU1tqb2JfaWRdCgoKQGFwcC5nZXQoIi9hcGkvam9icy97am9iX2lkfSIpCmRlZiBqb2Jfc3RhdHVzKGpvYl9pZDogc3RyKToKICAgIGdldF9qb2Ioam9iX2lkKQogICAgcmV0dXJuIGNsaXBub3ZhLnNuYXBzaG90KGpvYl9pZCkKCgpAYXBwLnBvc3QoIi9hcGkvam9icy97am9iX2lkfS9jbGlwcy97bn0iKQpkZWYgZWRpdF9jbGlwKGpvYl9pZDogc3RyLCBuOiBpbnQsIHN0eWxlOiBzdHIgPSBGb3JtKCJ7fSIpLCBtdXNpYzogVXBsb2FkRmlsZSB8IE5vbmUgPSBGaWxlKE5vbmUpKToKICAgIGpvYiA9IGdldF9qb2Ioam9iX2lkKQogICAgaWYgbm90IDEgPD0gbiA8PSBsZW4oam9iWyJyYXciXSk6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbig0MDQsICJDbGlwIGluY29ubnUuIikKICAgIGlmIGpvYlsicmF3Il1bbiAtIDFdLmdldCgiYnVzeSIpOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oNDA5LCAiQ2UgY2xpcCBlc3QgZMOpasOgIGVuIGNvdXJzIGRlIG1vbnRhZ2UuIikKICAgIGFsbG93ZWQgPSBjbGlwbm92YS5ERUZBVUxUX1NUWUxFLmtleXMoKSB8IHsibXVzaWNfb24ifQogICAgZGF0YSA9IHtrOiB2IGZvciBrLCB2IGluIGpzb24ubG9hZHMoc3R5bGUpLml0ZW1zKCkgaWYgayBpbiBhbGxvd2VkfQogICAgdHJhY2sgPSBOb25lCiAgICBpZiBtdXNpYyBhbmQgbXVzaWMuZmlsZW5hbWU6CiAgICAgICAgdHJhY2sgPSBjbGlwbm92YS5XT1JLIC8gam9iX2lkIC8gKCJtdXNpYyIgKyAoY2xpcG5vdmEuUGF0aChtdXNpYy5maWxlbmFtZSkuc3VmZml4IG9yICIubXAzIikpCiAgICAgICAgd2l0aCB0cmFjay5vcGVuKCJ3YiIpIGFzIGY6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihtdXNpYy5maWxlLCBmKQogICAgY2xpcG5vdmEuZWRpdF9jbGlwKGpvYl9pZCwgbiwgZGF0YSwgdHJhY2spCiAgICByZXR1cm4geyJvayI6IFRydWV9CgoKQGFwcC5nZXQoIi9hcGkvam9icy97am9iX2lkfS96aXAiKQpkZWYgZG93bmxvYWRfYWxsKGpvYl9pZDogc3RyKToKICAgICIiIlRvdXMgbGVzIGNsaXBzIGRhbnMgdW4gc2V1bCBmaWNoaWVyIFpJUC4iIiIKICAgIGpvYiA9IGdldF9qb2Ioam9iX2lkKQogICAgYnVmID0gaW8uQnl0ZXNJTygpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShidWYsICJ3IiwgemlwZmlsZS5aSVBfU1RPUkVEKSBhcyB6OgogICAgICAgIGZvciBjIGluIGpvYlsicmF3Il06CiAgICAgICAgICAgIHoud3JpdGUoY2xpcG5vdmEuV09SSyAvIGpvYl9pZCAvIGNbIm5hbWUiXSwgY1sibmFtZSJdKQogICAgYnVmLnNlZWsoMCkKICAgIHJldHVybiBTdHJlYW1pbmdSZXNwb25zZShidWYsIG1lZGlhX3R5cGU9ImFwcGxpY2F0aW9uL3ppcCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaGVhZGVycz17IkNvbnRlbnQtRGlzcG9zaXRpb24iOiBmJ2F0dGFjaG1lbnQ7IGZpbGVuYW1lPSJjbGlwbm92YS17am9iX2lkfS56aXAiJ30pCg==", "requirements.txt": "ZmFzdGFwaT49MC4xMTAKdXZpY29ybj49MC4yOQpweXRob24tbXVsdGlwYXJ0Pj0wLjAuOQp5dC1kbHA+PTIwMjQuMTAuMApmYXN0ZXItd2hpc3Blcj49MS4wCm9wZW5jdi1weXRob24taGVhZGxlc3M+PTQuOSw8NQo=", "static/index.html": "PCFkb2N0eXBlIGh0bWw+CjxodG1sIGxhbmc9ImZyIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xIj4KPHRpdGxlPkNsaXBOb3ZhPC90aXRsZT4KPGxpbmsgcmVsPSJzdHlsZXNoZWV0IiBocmVmPSIvc3RhdGljL3N0eWxlLmNzcyI+CjwvaGVhZD4KPGJvZHk+CjxtYWluIHN0eWxlPSJtYXgtd2lkdGg6NzYwcHgiPgogIDxoZWFkZXI+PGgxPkNsaXA8c3Bhbj5Ob3ZhPC9zcGFuPjwvaDE+PC9oZWFkZXI+CgogIDwhLS0gw4lUQVBFIDEgOiBsaWVuIC0tPgogIDxzZWN0aW9uIGlkPSJzdGVwMSIgc3R5bGU9ImRpc3BsYXk6Z3JpZDtnYXA6MTZweCI+CiAgICA8ZGl2IGNsYXNzPSJoZXJvIj4KICAgICAgPHAgY2xhc3M9Im5vdGUiPkfDiU7DiVJBVEVVUiBERSBDTElQUyBQQVIgSUEgwrcgMTAwICUgR1JBVFVJVDwvcD4KICAgICAgPGgyPjEgdmlkw6lvIGxvbmd1ZSwgMTAgY2xpcHMgdmlyYXV4LjwvaDI+CiAgICAgIDxwIGNsYXNzPSJzdWIiPkNsaXBOb3ZhIHRyYW5zZm9ybWUgdGVzIHZpZMOpb3MgbG9uZ3VlcyBlbiBTaG9ydHMsIFJlZWxzIGV0IFRpa1Rva3Mgc291cy10aXRyw6lzLjwvcD4KICAgIDwvZGl2PgogICAgPGZvcm0gY2xhc3M9ImRyb3AiIGlkPSJsaW5rRm9ybSI+CiAgICAgIDxpbnB1dCBpZD0idXJsIiB0eXBlPSJ1cmwiIHBsYWNlaG9sZGVyPSLwn5SXIENvbGxlIHVuIGxpZW4gdmlkw6lvIChZb3VUdWJlLCBUaWtUb2vigKYpIiBhcmlhLWxhYmVsPSJMaWVuIHZpZMOpbyI+CiAgICAgIDxidXR0b24+T2J0ZW5pciBsZXMgY2xpcHM8L2J1dHRvbj4KICAgIDwvZm9ybT4KICAgIDxkaXYgY2xhc3M9Im9yIj5vdTwvZGl2PgogICAgPGxhYmVsIGNsYXNzPSJidG4gZ2hvc3QiIGZvcj0iZmlsZSIgc3R5bGU9ImN1cnNvcjpwb2ludGVyIj7irIYgSW1wb3J0ZXIgdW4gZmljaGllciB2aWTDqW88aW5wdXQgaWQ9ImZpbGUiIHR5cGU9ImZpbGUiIGFjY2VwdD0idmlkZW8vKiIgaGlkZGVuPjwvbGFiZWw+CiAgICA8ZGl2IGlkPSJlcnIxIiBjbGFzcz0iZXJyIiBhcmlhLWxpdmU9InBvbGl0ZSI+PC9kaXY+CiAgICA8ZGl2IGNsYXNzPSJ0b29scyI+CiAgICAgIDxkaXYgY2xhc3M9InRvb2wiPjxiPuKcgu+4jzwvYj5Mb25nIOKGkiBjb3VydHM8L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0idG9vbCI+PGI+Q0M8L2I+U291cy10aXRyZXMgSUE8L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0idG9vbCI+PGI+8J+OrDwvYj7DiWRpdGV1ciB2aWTDqW88L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0idG9vbCI+PGI+8J+OtTwvYj5NdXNpcXVlPC9kaXY+CiAgICA8L2Rpdj4KICAgIDxoMj5NZXMgcHJvamV0czwvaDI+CiAgICA8ZGl2IGNsYXNzPSJwcm9qZWN0cyIgaWQ9InByb2plY3RzIj48cCBjbGFzcz0ibm90ZSI+QXVjdW4gcHJvamV0IHBvdXIgbCdpbnN0YW50LjwvcD48L2Rpdj4KICA8L3NlY3Rpb24+CgogIDwhLS0gw4lUQVBFIDIgOiByw6lnbGFnZXMgLS0+CiAgPGZvcm0gaWQ9InN0ZXAyIiBjbGFzcz0iY2FyZCIgaGlkZGVuPgogICAgPGltZyBpZD0idGh1bWIiIGNsYXNzPSJ0aHVtYiIgYWx0PSIiPgogICAgPHZpZGVvIGlkPSJsb2NhbFByZXZpZXciIGNsYXNzPSJ0aHVtYiIgY29udHJvbHMgcGxheXNpbmxpbmUgaGlkZGVuPjwvdmlkZW8+CiAgICA8YiBpZD0idlRpdGxlIj48L2I+CiAgICA8cCBjbGFzcz0ibm90ZSI+VXRpbGlzZXIgdW5lIHZpZMOpbyBxdWkgbmUgdCdhcHBhcnRpZW50IHBhcyBwZXV0IHZpb2xlciBsZSBkcm9pdCBkJ2F1dGV1ci4gRW4gY29udGludWFudCwgdHUgY29uZmlybWVzIHF1ZSBjJ2VzdCB0b24gY29udGVudSBvdSBxdWUgdHUgYXMgbCdhdXRvcmlzYXRpb24uPC9wPgoKICAgIDxkaXYgY2xhc3M9InNlZyIgaWQ9ImFpU2VnIj48YnV0dG9uIHR5cGU9ImJ1dHRvbiIgY2xhc3M9Im9uIiBkYXRhLWFpPSIxIj5Ew6ljb3VwYWdlIElBPC9idXR0b24+PGJ1dHRvbiB0eXBlPSJidXR0b24iIGRhdGEtYWk9IjAiPk5lIHBhcyBkw6ljb3VwZXI8L2J1dHRvbj48L2Rpdj4KCiAgICA8ZGl2IGNsYXNzPSJncmlkIiBpZD0iYWlPcHRzIj4KICAgICAgPGxhYmVsIGZvcj0ibGVuZ3RoIj5EdXLDqWUgZGVzIGNsaXBzCiAgICAgICAgPHNlbGVjdCBpZD0ibGVuZ3RoIj48b3B0aW9uIHZhbHVlPSIxNS05MCI+QXV0byAoMC0xbTMwKTwvb3B0aW9uPjxvcHRpb24gdmFsdWU9IjEwLTMwIj4mbHQ7IDMwIHM8L29wdGlvbj48b3B0aW9uIHZhbHVlPSIzMC02MCIgc2VsZWN0ZWQ+MzAgcyDigJMgNjAgczwvb3B0aW9uPgogICAgICAgICAgPG9wdGlvbiB2YWx1ZT0iNjAtOTAiPjYwIHMg4oCTIDkwIHM8L29wdGlvbj48b3B0aW9uIHZhbHVlPSI5MC0xODAiPjkwIHMg4oCTIDMgbWluPC9vcHRpb24+PC9zZWxlY3Q+PC9sYWJlbD4KICAgICAgPGxhYmVsIGZvcj0iY291bnQiPk5vbWJyZSBkZSBjbGlwczxpbnB1dCBpZD0iY291bnQiIHR5cGU9Im51bWJlciIgdmFsdWU9IjEwIiBtaW49IjEiIG1heD0iMzAiPjwvbGFiZWw+CiAgICAgIDxsYWJlbCBmb3I9ImxhbmciPkxhbmd1ZQogICAgICAgIDxzZWxlY3QgaWQ9ImxhbmciPjxvcHRpb24gdmFsdWU9ImZyIj5GcmFuw6dhaXM8L29wdGlvbj48b3B0aW9uIHZhbHVlPSJhciI+QXJhYmUgLyBEYXJpamE8L29wdGlvbj48b3B0aW9uIHZhbHVlPSJlbiI+QW5nbGFpczwvb3B0aW9uPgogICAgICAgICAgPG9wdGlvbiB2YWx1ZT0iZXMiPkVzcGFnbm9sPC9vcHRpb24+PG9wdGlvbiB2YWx1ZT0iYXV0byI+QXV0bzwvb3B0aW9uPjwvc2VsZWN0PjwvbGFiZWw+CiAgICAgIDxsYWJlbCBmb3I9Im1vZGVsIj5QcsOpY2lzaW9uIGRlcyBzb3VzLXRpdHJlcwogICAgICAgIDxzZWxlY3QgaWQ9Im1vZGVsIj48b3B0aW9uIHZhbHVlPSJiYXNlIj5SYXBpZGU8L29wdGlvbj48b3B0aW9uIHZhbHVlPSJzbWFsbCIgc2VsZWN0ZWQ+Tm9ybWFsZTwvb3B0aW9uPjxvcHRpb24gdmFsdWU9Im1lZGl1bSI+SGF1dGUgKGxlbnQpPC9vcHRpb24+PC9zZWxlY3Q+PC9sYWJlbD4KICAgIDwvZGl2PgogICAgPGxhYmVsIGZvcj0ia2V5d29yZHMiPkluY2x1cmUgZGVzIG1vbWVudHMgcHLDqWNpcyA8c21hbGwgc3R5bGU9ImRpc3BsYXk6aW5saW5lIj4obW90cy1jbMOpcyBzw6lwYXLDqXMgcGFyIGRlcyB2aXJndWxlcyk8L3NtYWxsPgogICAgICA8aW5wdXQgaWQ9ImtleXdvcmRzIiBwbGFjZWhvbGRlcj0iRXguIDogYXJnZW50LCBlcnJldXIsIHNlY3JldCI+PC9sYWJlbD4KCiAgICA8ZGl2PgogICAgICA8bGFiZWw+UGxhZ2UgZGUgdHJhaXRlbWVudDwvbGFiZWw+CiAgICAgIDxkaXYgY2xhc3M9ImR1YWwiPjxkaXYgY2xhc3M9InRyYWNrIj48L2Rpdj48ZGl2IGNsYXNzPSJmaWxsIiBpZD0iZmlsbCI+PC9kaXY+CiAgICAgICAgPGlucHV0IGlkPSJ0RnJvbSIgdHlwZT0icmFuZ2UiIG1pbj0iMCIgbWF4PSIxMDAiIHZhbHVlPSIwIiBhcmlhLWxhYmVsPSJEw6lidXQiPgogICAgICAgIDxpbnB1dCBpZD0idFRvIiB0eXBlPSJyYW5nZSIgbWluPSIwIiBtYXg9IjEwMCIgdmFsdWU9IjEwMCIgYXJpYS1sYWJlbD0iRmluIj48L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0idGltZXMiPjxzcGFuIGlkPSJ0RnJvbUwiPjAwOjAwPC9zcGFuPjxzcGFuIGlkPSJ0VG9MIj4tLTotLTwvc3Bhbj48L2Rpdj4KICAgIDwvZGl2PgoKICAgIDxkaXYgY2xhc3M9ImdyaWQiPgogICAgICA8bGFiZWwgY2xhc3M9ImNoZWNrIiBmb3I9InN1YnMiPjxpbnB1dCBpZD0ic3VicyIgdHlwZT0iY2hlY2tib3giIGNoZWNrZWQ+IFNvdXMtdGl0cmVzIElBPC9sYWJlbD4KICAgICAgPGxhYmVsIGZvcj0iY29sb3IiPkNvdWxldXIgZHUgbW90IHByb25vbmPDqTxpbnB1dCBpZD0iY29sb3IiIHR5cGU9ImNvbG9yIiB2YWx1ZT0iI2ZmZTUwMCI+PC9sYWJlbD4KICAgIDwvZGl2PgogICAgPGxhYmVsIGNsYXNzPSJjaGVjayIgZm9yPSJyaWdodHMiPjxpbnB1dCBpZD0icmlnaHRzIiB0eXBlPSJjaGVja2JveCIgcmVxdWlyZWQ+IEplIHBvc3PDqGRlIGxlcyBkcm9pdHMgZGUgY2V0dGUgdmlkw6lvIChvdSBqJ2FpIGwnYXV0b3Jpc2F0aW9uKTwvbGFiZWw+CiAgICA8ZGl2IGNsYXNzPSJyb3ciPgogICAgICA8YnV0dG9uIHR5cGU9ImJ1dHRvbiIgY2xhc3M9Imdob3N0IiBpZD0iYmFjayI+4oaQIFJldG91cjwvYnV0dG9uPgogICAgICA8YnV0dG9uIGlkPSJnbyI+T2J0ZW5pciBsZXMgY2xpcHMgZW4gMSBjbGljPC9idXR0b24+CiAgICA8L2Rpdj4KICAgIDxkaXYgaWQ9ImVycjIiIGNsYXNzPSJlcnIiIGFyaWEtbGl2ZT0icG9saXRlIj48L2Rpdj4KICA8L2Zvcm0+CjwvbWFpbj4KPHNjcmlwdD4KY29uc3QgJCA9IGlkID0+IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKGlkKTsKbGV0IGR1ciA9IDAsIGFpID0gdHJ1ZSwgc291cmNlID0gbnVsbDsgLy8gc291cmNlID0ge3VybH0gb3Uge2ZpbGV9CmNvbnN0IGZtdCA9IHMgPT4geyBzID0gTWF0aC5yb3VuZChzKTsgY29uc3QgaCA9IE1hdGguZmxvb3IocyAvIDM2MDApLCBtID0gTWF0aC5mbG9vcihzICUgMzYwMCAvIDYwKSwgeCA9IHMgJSA2MDsKICByZXR1cm4gKGggPyBoICsgJzonIDogJycpICsgU3RyaW5nKG0pLnBhZFN0YXJ0KDIsICcwJykgKyAnOicgKyBTdHJpbmcoeCkucGFkU3RhcnQoMiwgJzAnKTsgfTsKCmZ1bmN0aW9uIHNob3dTdGVwMigpIHsgJCgnc3RlcDEnKS5oaWRkZW4gPSB0cnVlOyAkKCdzdGVwMicpLmhpZGRlbiA9IGZhbHNlOyB1cGRSYW5nZSgpOyB9CiQoJ2xpbmtGb3JtJykuYWRkRXZlbnRMaXN0ZW5lcignc3VibWl0JywgYXN5bmMgZSA9PiB7CiAgZS5wcmV2ZW50RGVmYXVsdCgpOwogIGNvbnN0IHVybCA9ICQoJ3VybCcpLnZhbHVlLnRyaW0oKTsgaWYgKCF1cmwpIHJldHVybjsKICAkKCdlcnIxJykudGV4dENvbnRlbnQgPSAnTGVjdHVyZSBkdSBsaWVu4oCmJzsKICB0cnkgewogICAgY29uc3QgciA9IGF3YWl0IGZldGNoKCcvYXBpL2luZm8/dXJsPScgKyBlbmNvZGVVUklDb21wb25lbnQodXJsKSk7IGNvbnN0IGQgPSBhd2FpdCByLmpzb24oKTsKICAgIGlmICghci5vaykgdGhyb3cgbmV3IEVycm9yKGQuZGV0YWlsKTsKICAgIHNvdXJjZSA9IHsgdXJsIH07IGR1ciA9IGQuZHVyYXRpb24gfHwgMDsKICAgICQoJ3RodW1iJykuaGlkZGVuID0gIWQudGh1bWJuYWlsOyAkKCd0aHVtYicpLnNyYyA9IGQudGh1bWJuYWlsIHx8ICcnOyAkKCdsb2NhbFByZXZpZXcnKS5oaWRkZW4gPSB0cnVlOwogICAgJCgndlRpdGxlJykudGV4dENvbnRlbnQgPSBkLnRpdGxlOyAkKCdlcnIxJykudGV4dENvbnRlbnQgPSAnJzsgc2hvd1N0ZXAyKCk7CiAgfSBjYXRjaCAoZXJyKSB7ICQoJ2VycjEnKS50ZXh0Q29udGVudCA9IGVyci5tZXNzYWdlOyB9Cn0pOwokKCdmaWxlJykuYWRkRXZlbnRMaXN0ZW5lcignY2hhbmdlJywgKCkgPT4gewogIGNvbnN0IGYgPSAkKCdmaWxlJykuZmlsZXNbMF07IGlmICghZikgcmV0dXJuOwogIHNvdXJjZSA9IHsgZmlsZTogZiB9OwogIGNvbnN0IHYgPSAkKCdsb2NhbFByZXZpZXcnKTsgdi5zcmMgPSBVUkwuY3JlYXRlT2JqZWN0VVJMKGYpOyB2LmhpZGRlbiA9IGZhbHNlOyAkKCd0aHVtYicpLmhpZGRlbiA9IHRydWU7CiAgdi5vbmxvYWRlZG1ldGFkYXRhID0gKCkgPT4geyBkdXIgPSB2LmR1cmF0aW9uIHx8IDA7IHVwZFJhbmdlKCk7IH07CiAgJCgndlRpdGxlJykudGV4dENvbnRlbnQgPSBmLm5hbWU7IHNob3dTdGVwMigpOwp9KTsKJCgnYmFjaycpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywgKCkgPT4geyAkKCdzdGVwMicpLmhpZGRlbiA9IHRydWU7ICQoJ3N0ZXAxJykuaGlkZGVuID0gZmFsc2U7IH0pOwokKCdhaVNlZycpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywgZSA9PiB7CiAgY29uc3QgYiA9IGUudGFyZ2V0LmNsb3Nlc3QoJ2J1dHRvbicpOyBpZiAoIWIpIHJldHVybjsKICBhaSA9IGIuZGF0YXNldC5haSA9PT0gJzEnOwogICQoJ2FpU2VnJykucXVlcnlTZWxlY3RvckFsbCgnYnV0dG9uJykuZm9yRWFjaCh4ID0+IHguY2xhc3NMaXN0LnRvZ2dsZSgnb24nLCB4ID09PSBiKSk7CiAgJCgnYWlPcHRzJykuc3R5bGUub3BhY2l0eSA9IGFpID8gMSA6IC40Owp9KTsKZnVuY3Rpb24gdXBkUmFuZ2UoKSB7CiAgbGV0IGEgPSArJCgndEZyb20nKS52YWx1ZSwgYiA9ICskKCd0VG8nKS52YWx1ZTsKICBpZiAoYSA+IGIgLSAxKSB7IGEgPSBiIC0gMTsgJCgndEZyb20nKS52YWx1ZSA9IGE7IH0KICAkKCdmaWxsJykuc3R5bGUubGVmdCA9IGEgKyAnJSc7ICQoJ2ZpbGwnKS5zdHlsZS53aWR0aCA9IChiIC0gYSkgKyAnJSc7CiAgJCgndEZyb21MJykudGV4dENvbnRlbnQgPSBmbXQoZHVyICogYSAvIDEwMCk7ICQoJ3RUb0wnKS50ZXh0Q29udGVudCA9IGR1ciA/IGZtdChkdXIgKiBiIC8gMTAwKSA6ICctLTotLSc7Cn0KJCgndEZyb20nKS5hZGRFdmVudExpc3RlbmVyKCdpbnB1dCcsIHVwZFJhbmdlKTsgJCgndFRvJykuYWRkRXZlbnRMaXN0ZW5lcignaW5wdXQnLCB1cGRSYW5nZSk7CgokKCdzdGVwMicpLmFkZEV2ZW50TGlzdGVuZXIoJ3N1Ym1pdCcsIGFzeW5jIGUgPT4gewogIGUucHJldmVudERlZmF1bHQoKTsgJCgnZ28nKS5kaXNhYmxlZCA9IHRydWU7ICQoJ2VycjInKS50ZXh0Q29udGVudCA9ICdFbnZvaeKApic7CiAgY29uc3QgZmQgPSBuZXcgRm9ybURhdGEoKTsKICBpZiAoc291cmNlLnVybCkgZmQuc2V0KCd1cmwnLCBzb3VyY2UudXJsKTsgZWxzZSBmZC5zZXQoJ2ZpbGUnLCBzb3VyY2UuZmlsZSk7CiAgY29uc3QgW21uLCBteF0gPSAkKCdsZW5ndGgnKS52YWx1ZS5zcGxpdCgnLScpOwogIGZvciAoY29uc3QgW2ssIHZdIG9mIE9iamVjdC5lbnRyaWVzKHsgbWluX2Q6IG1uLCBtYXhfZDogbXgsIGNvdW50OiAkKCdjb3VudCcpLnZhbHVlLCBsYW5nOiAkKCdsYW5nJykudmFsdWUsIG1vZGVsOiAkKCdtb2RlbCcpLnZhbHVlLAogICAga2V5d29yZHM6ICQoJ2tleXdvcmRzJykudmFsdWUsIHJpZ2h0czogJCgncmlnaHRzJykuY2hlY2tlZCwgc3ViczogJCgnc3VicycpLmNoZWNrZWQsIGFpLCBjb2xvcjogJCgnY29sb3InKS52YWx1ZS5zbGljZSgxKSwKICAgIHRfZnJvbTogZHVyICogJCgndEZyb20nKS52YWx1ZSAvIDEwMCwgdF90bzogKyQoJ3RUbycpLnZhbHVlID49IDEwMCA/IDAgOiBkdXIgKiAkKCd0VG8nKS52YWx1ZSAvIDEwMCB9KSkgZmQuc2V0KGssIHYpOwogIHRyeSB7CiAgICBjb25zdCByID0gYXdhaXQgZmV0Y2goJy9hcGkvam9icycsIHsgbWV0aG9kOiAnUE9TVCcsIGJvZHk6IGZkIH0pOyBjb25zdCBkID0gYXdhaXQgci5qc29uKCk7CiAgICBpZiAoIXIub2spIHRocm93IG5ldyBFcnJvcihkLmRldGFpbCk7CiAgICBsb2NhdGlvbi5ocmVmID0gJy9jbGlwcz9qb2I9JyArIGQuaWQ7CiAgfSBjYXRjaCAoZXJyKSB7ICQoJ2VycjInKS50ZXh0Q29udGVudCA9IGVyci5tZXNzYWdlOyAkKCdnbycpLmRpc2FibGVkID0gZmFsc2U7IH0KfSk7CgpmZXRjaCgnL2FwaS9wcm9qZWN0cycpLnRoZW4ociA9PiByLmpzb24oKSkudGhlbihsaXN0ID0+IHsKICBpZiAoIWxpc3QubGVuZ3RoKSByZXR1cm47CiAgJCgncHJvamVjdHMnKS5pbm5lckhUTUwgPSBsaXN0Lm1hcChwID0+IGA8YSBjbGFzcz0icHJvamVjdCBjYXJkIiBocmVmPSIvY2xpcHM/am9iPSR7cC5pZH0iPgogICAgJHtwLnRodW1iID8gYDx2aWRlbyBzcmM9IiR7cC50aHVtYi5zcGxpdCgnPycpWzBdfSN0PTAuNSIgbXV0ZWQgcHJlbG9hZD0ibWV0YWRhdGEiPjwvdmlkZW8+YCA6ICcnfQogICAgPGIgc3R5bGU9Im92ZXJmbG93LXdyYXA6YW55d2hlcmUiPiR7KHAubmFtZSB8fCAnUHJvamV0JykucmVwbGFjZSgvPC9nLCAnJmx0OycpLnNsaWNlKDAsIDYwKX08L2I+CiAgICA8c21hbGw+JHtwLnN0YXR1cyA9PT0gJ3J1bm5pbmcnID8gJ+KPsyAnICsgcC5zdGVwLnJlcGxhY2UoLzwvZywgJyZsdDsnKSA6IHAuY291bnQgKyAnIGNsaXBzJ308L3NtYWxsPjwvYT5gKS5qb2luKCcnKTsKfSk7Cjwvc2NyaXB0Pgo8L2JvZHk+CjwvaHRtbD4K", "static/clips.html": "PCFkb2N0eXBlIGh0bWw+CjxodG1sIGxhbmc9ImZyIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xIj4KPHRpdGxlPkNsaXBOb3ZhIOKAlCBNZXMgY2xpcHM8L3RpdGxlPgo8bGluayByZWw9InN0eWxlc2hlZXQiIGhyZWY9Ii9zdGF0aWMvc3R5bGUuY3NzIj4KPC9oZWFkPgo8Ym9keT4KPG1haW4+CiAgPGhlYWRlciBjbGFzcz0icm93IiBzdHlsZT0iYWxpZ24taXRlbXM6Y2VudGVyIj4KICAgIDxoMT48YSBocmVmPSIvIj5DbGlwPHNwYW4+Tm92YTwvc3Bhbj48L2E+PC9oMT4KICAgIDxhIGNsYXNzPSJidG4gZ2hvc3QiIGhyZWY9Ii8iIHN0eWxlPSJmbGV4OjAiPisgTm91dmVsbGUgdmlkw6lvPC9hPgogIDwvaGVhZGVyPgoKICA8ZGl2IGNsYXNzPSJjYXJkIiBpZD0icHJvZ3Jlc3MiPgogICAgPGRpdiBpZD0ic3RhdHVzIiBhcmlhLWxpdmU9InBvbGl0ZSI+Q2hhcmdlbWVudOKApjwvZGl2PjxkaXYgY2xhc3M9ImJhciI+PGkgaWQ9ImJhciI+PC9pPjwvZGl2PgogIDwvZGl2PgoKICA8c2VjdGlvbiBjbGFzcz0iY2FyZCIgaWQ9ImVkaXRvckNhcmQiIGhpZGRlbj4KICAgIDxoMiBpZD0iZWRUaXRsZSI+TW9udGFnZTwvaDI+CiAgICA8ZGl2IGNsYXNzPSJlZGl0b3IiPgogICAgICA8ZGl2IGNsYXNzPSJwcmV2aWV3Ij4KICAgICAgICA8dmlkZW8gaWQ9ImVkVmlkZW8iIGNvbnRyb2xzIHBsYXlzaW5saW5lPjwvdmlkZW8+CiAgICAgICAgPHNtYWxsIGlkPSJlZEluZm8iPjwvc21hbGw+CiAgICAgICAgPGRpdiBjbGFzcz0icm93IiBzdHlsZT0ibWFyZ2luLXRvcDo4cHgiPgogICAgICAgICAgPGEgY2xhc3M9ImJ0biIgaWQ9ImVkRG93bmxvYWQiIGRvd25sb2FkPuKshyBUw6lsw6ljaGFyZ2VyPC9hPgogICAgICAgIDwvZGl2PgogICAgICAgIDxzbWFsbCBzdHlsZT0ibWFyZ2luLXRvcDo2cHgiPlN1ciBpUGFkL2lQaG9uZSA6IHRvdWNoZSBUw6lsw6ljaGFyZ2VyLCBvdXZyZSBsZSBmaWNoaWVyLCBwdWlzIFBhcnRhZ2VyIOKGkiDCqyBFbnJlZ2lzdHJlciBsYSB2aWTDqW8gwrsgcG91ciBsJ2F2b2lyIGRhbnMgUGhvdG9zLjwvc21hbGw+CiAgICAgIDwvZGl2PgoKICAgICAgPGZvcm0gaWQ9ImVkRm9ybSIgc3R5bGU9ImRpc3BsYXk6Z3JpZDtnYXA6MTRweCI+CiAgICAgICAgPGRpdiBjbGFzcz0idGFicyIgaWQ9InRhYnMiIHJvbGU9InRhYmxpc3QiPgogICAgICAgICAgPGJ1dHRvbiB0eXBlPSJidXR0b24iIHJvbGU9InRhYiIgYXJpYS1zZWxlY3RlZD0idHJ1ZSIgZGF0YS10YWI9InN1YnMiIGNsYXNzPSJvbiI+U291cy10aXRyZXM8L2J1dHRvbj4KICAgICAgICAgIDxidXR0b24gdHlwZT0iYnV0dG9uIiByb2xlPSJ0YWIiIGFyaWEtc2VsZWN0ZWQ9ImZhbHNlIiBkYXRhLXRhYj0ic3R5bGUiPlN0eWxlPC9idXR0b24+CiAgICAgICAgICA8YnV0dG9uIHR5cGU9ImJ1dHRvbiIgcm9sZT0idGFiIiBhcmlhLXNlbGVjdGVkPSJmYWxzZSIgZGF0YS10YWI9Im11c2ljIj5NdXNpcXVlICYgc29uPC9idXR0b24+CiAgICAgICAgICA8YnV0dG9uIHR5cGU9ImJ1dHRvbiIgcm9sZT0idGFiIiBhcmlhLXNlbGVjdGVkPSJmYWxzZSIgZGF0YS10YWI9ImN1dCI+Q291cGUgJiBjYWRyYWdlPC9idXR0b24+CiAgICAgICAgICA8YnV0dG9uIHR5cGU9ImJ1dHRvbiIgcm9sZT0idGFiIiBhcmlhLXNlbGVjdGVkPSJmYWxzZSIgZGF0YS10YWI9InRleHQiPlRleHRlIGQnYWNjcm9jaGU8L2J1dHRvbj4KICAgICAgICA8L2Rpdj4KCiAgICAgICAgPGRpdiBkYXRhLXBhbmVsPSJzdWJzIiBjbGFzcz0iZ3JpZCI+CiAgICAgICAgICA8bGFiZWwgY2xhc3M9ImNoZWNrIiBmb3I9InNfc3VicyI+PGlucHV0IGlkPSJzX3N1YnMiIHR5cGU9ImNoZWNrYm94Ij4gQWZmaWNoZXIgbGVzIHNvdXMtdGl0cmVzPC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBjbGFzcz0iY2hlY2siIGZvcj0ic191cHBlciI+PGlucHV0IGlkPSJzX3VwcGVyIiB0eXBlPSJjaGVja2JveCI+IE1BSlVTQ1VMRVM8L2xhYmVsPgogICAgICAgICAgPGxhYmVsIGZvcj0ic193b3JkcyI+TW90cyDDoCBsYSBmb2lzPHNlbGVjdCBpZD0ic193b3JkcyI+PG9wdGlvbj4xPC9vcHRpb24+PG9wdGlvbj4yPC9vcHRpb24+PG9wdGlvbj4zPC9vcHRpb24+PG9wdGlvbj40PC9vcHRpb24+PG9wdGlvbj42PC9vcHRpb24+PC9zZWxlY3Q+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBmb3I9InNfcG9zaXRpb24iPlBvc2l0aW9uPHNlbGVjdCBpZD0ic19wb3NpdGlvbiI+PG9wdGlvbiB2YWx1ZT0iaGF1dCI+SGF1dDwvb3B0aW9uPjxvcHRpb24gdmFsdWU9Im1pbGlldSI+TWlsaWV1PC9vcHRpb24+PG9wdGlvbiB2YWx1ZT0iYmFzIj5CYXM8L29wdGlvbj48L3NlbGVjdD48L2xhYmVsPgogICAgICAgIDwvZGl2PgoKICAgICAgICA8ZGl2IGRhdGEtcGFuZWw9InN0eWxlIiBjbGFzcz0iZ3JpZCIgaGlkZGVuPgogICAgICAgICAgPGRpdiBjbGFzcz0icHJlc2V0cyIgaWQ9InByZXNldHMiPgogICAgICAgICAgICA8YnV0dG9uIHR5cGU9ImJ1dHRvbiIgZGF0YS1wPSJqYXVuZSI+R3JvcyBqYXVuZTwvYnV0dG9uPjxidXR0b24gdHlwZT0iYnV0dG9uIiBkYXRhLXA9ImJvaXRlIj5Cb8OudGUgbm9pcmU8L2J1dHRvbj4KICAgICAgICAgICAgPGJ1dHRvbiB0eXBlPSJidXR0b24iIGRhdGEtcD0ibmVvbiI+TsOpb248L2J1dHRvbj48YnV0dG9uIHR5cGU9ImJ1dHRvbiIgZGF0YS1wPSJtaW5pbWFsIj5NaW5pbWFsPC9idXR0b24+PC9kaXY+CiAgICAgICAgICA8bGFiZWwgZm9yPSJzX2ZvbnQiPlBvbGljZTxzZWxlY3QgaWQ9InNfZm9udCI+CiAgICAgICAgICAgIDxvcHRpb24+RGVqYVZ1IFNhbnM8L29wdGlvbj48b3B0aW9uPkFyaWFsPC9vcHRpb24+PG9wdGlvbj5JbXBhY3Q8L29wdGlvbj48b3B0aW9uPk1vbnRzZXJyYXQ8L29wdGlvbj4KICAgICAgICAgICAgPG9wdGlvbj5CZWJhcyBOZXVlPC9vcHRpb24+PG9wdGlvbj5Qb3BwaW5zPC9vcHRpb24+PG9wdGlvbj5Db21pYyBTYW5zIE1TPC9vcHRpb24+PC9zZWxlY3Q+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBmb3I9InNfc2l6ZSI+VGFpbGxlIDxzcGFuIGlkPSJ2X3NpemUiPjwvc3Bhbj48aW5wdXQgaWQ9InNfc2l6ZSIgdHlwZT0icmFuZ2UiIG1pbj0iNDAiIG1heD0iMTQwIiBzdGVwPSIyIj48L2xhYmVsPgogICAgICAgICAgPGxhYmVsIGZvcj0ic19jb2xvciI+Q291bGV1ciBkdSB0ZXh0ZTxpbnB1dCBpZD0ic19jb2xvciIgdHlwZT0iY29sb3IiPjwvbGFiZWw+CiAgICAgICAgICA8bGFiZWwgZm9yPSJzX2hpZ2hsaWdodCI+Q291bGV1ciBkdSBtb3QgcHJvbm9uY8OpPGlucHV0IGlkPSJzX2hpZ2hsaWdodCIgdHlwZT0iY29sb3IiPjwvbGFiZWw+CiAgICAgICAgICA8bGFiZWwgZm9yPSJzX291dGxpbmUiPkNvbnRvdXIgPHNwYW4gaWQ9InZfb3V0bGluZSI+PC9zcGFuPjxpbnB1dCBpZD0ic19vdXRsaW5lIiB0eXBlPSJyYW5nZSIgbWluPSIwIiBtYXg9IjEyIj48L2xhYmVsPgogICAgICAgICAgPGxhYmVsIGNsYXNzPSJjaGVjayIgZm9yPSJzX2JveCI+PGlucHV0IGlkPSJzX2JveCIgdHlwZT0iY2hlY2tib3giPiBGb25kIG5vaXIgZGVycmnDqHJlIGxlIHRleHRlPC9sYWJlbD4KICAgICAgICA8L2Rpdj4KCiAgICAgICAgPGRpdiBkYXRhLXBhbmVsPSJtdXNpYyIgY2xhc3M9ImdyaWQiIGhpZGRlbj4KICAgICAgICAgIDxsYWJlbCBmb3I9Im11c2ljIj5NdXNpcXVlIGRlIGZvbmQgKE1QMywgbGlicmUgZGUgZHJvaXRzKTxpbnB1dCBpZD0ibXVzaWMiIHR5cGU9ImZpbGUiIGFjY2VwdD0iYXVkaW8vKiI+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBjbGFzcz0iY2hlY2siIGZvcj0ic19tdXNpY19vbiI+PGlucHV0IGlkPSJzX211c2ljX29uIiB0eXBlPSJjaGVja2JveCI+IE11c2lxdWUgYWN0aXbDqWUgKGTDqWNvY2hlIHBvdXIgbGEgcmV0aXJlcik8L2xhYmVsPgogICAgICAgICAgPGxhYmVsIGZvcj0ic19tdXNpY192b2x1bWUiPlZvbHVtZSBtdXNpcXVlIDxzcGFuIGlkPSJ2X211c2ljX3ZvbHVtZSI+PC9zcGFuPjxpbnB1dCBpZD0ic19tdXNpY192b2x1bWUiIHR5cGU9InJhbmdlIiBtaW49IjAiIG1heD0iMSIgc3RlcD0iMC4wNSI+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBmb3I9InNfdm9pY2Vfdm9sdW1lIj5Wb2x1bWUgdm9peCA8c3BhbiBpZD0idl92b2ljZV92b2x1bWUiPjwvc3Bhbj48aW5wdXQgaWQ9InNfdm9pY2Vfdm9sdW1lIiB0eXBlPSJyYW5nZSIgbWluPSIwIiBtYXg9IjIiIHN0ZXA9IjAuMSI+PC9sYWJlbD4KICAgICAgICAgIDxwIGNsYXNzPSJub3RlIiBzdHlsZT0iZ3JpZC1jb2x1bW46MS8tMSI+TXVzaXF1ZXMgZ3JhdHVpdGVzIDogQmlibGlvdGjDqHF1ZSBhdWRpbyBZb3VUdWJlLCBQaXhhYmF5IE11c2ljLiDDiXZpdGUgbGVzIGNoYW5zb25zIGNvbm51ZXMgOiBUaWtUb2sgZXQgWW91VHViZSBwZXV2ZW50IGJsb3F1ZXIgbGUgY2xpcC48L3A+CiAgICAgICAgPC9kaXY+CgogICAgICAgIDxkaXYgZGF0YS1wYW5lbD0iY3V0IiBjbGFzcz0iZ3JpZCIgaGlkZGVuPgogICAgICAgICAgPGxhYmVsIGZvcj0ic190cmltX3N0YXJ0Ij5Ew6lidXQgOiBjb3VwZXIgKHMpIDxzbWFsbD5uw6lnYXRpZiA9IHJhbGxvbmdlcjwvc21hbGw+PGlucHV0IGlkPSJzX3RyaW1fc3RhcnQiIHR5cGU9Im51bWJlciIgc3RlcD0iMC41IiBtaW49Ii0xMCIgbWF4PSIzMCI+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBmb3I9InNfdHJpbV9lbmQiPkZpbiA6IGNvdXBlciAocykgPHNtYWxsPm7DqWdhdGlmID0gcmFsbG9uZ2VyPC9zbWFsbD48aW5wdXQgaWQ9InNfdHJpbV9lbmQiIHR5cGU9Im51bWJlciIgc3RlcD0iMC41IiBtaW49Ii0xMCIgbWF4PSIzMCI+PC9sYWJlbD4KICAgICAgICAgIDxsYWJlbCBmb3I9InNfem9vbSI+Wm9vbSBzdXIgbGUgdmlzYWdlIDxzcGFuIGlkPSJ2X3pvb20iPjwvc3Bhbj48aW5wdXQgaWQ9InNfem9vbSIgdHlwZT0icmFuZ2UiIG1pbj0iMSIgbWF4PSIyIiBzdGVwPSIwLjA1Ij48L2xhYmVsPgogICAgICAgIDwvZGl2PgoKICAgICAgICA8ZGl2IGRhdGEtcGFuZWw9InRleHQiIGNsYXNzPSJncmlkIiBoaWRkZW4+CiAgICAgICAgICA8bGFiZWwgZm9yPSJzX3RpdGxlIiBzdHlsZT0iZ3JpZC1jb2x1bW46MS8tMSI+VGV4dGUgZml4ZSBlbiBoYXV0IGR1IGNsaXAgKGFjY3JvY2hlKTxpbnB1dCBpZD0ic190aXRsZSIgbWF4bGVuZ3RoPSI2MCIgcGxhY2Vob2xkZXI9IkV4LiA6IExhIHZyYWllIHJhaXNvbiBkZSB0b24gw6ljaGVjIPCfmLMiPjwvbGFiZWw+CiAgICAgICAgPC9kaXY+CgogICAgICAgIDxkaXYgY2xhc3M9InJvdyI+CiAgICAgICAgICA8YnV0dG9uIGlkPSJhcHBseSI+QXBwbGlxdWVyIGxlIG1vbnRhZ2U8L2J1dHRvbj4KICAgICAgICAgIDxidXR0b24gdHlwZT0iYnV0dG9uIiBjbGFzcz0iZ2hvc3QiIGlkPSJhcHBseUFsbCI+QXBwbGlxdWVyIGxlIHN0eWxlIMOgIHRvdXMgbGVzIGNsaXBzPC9idXR0b24+CiAgICAgICAgPC9kaXY+CiAgICAgICAgPGRpdiBpZD0iZWRTdGF0dXMiIGNsYXNzPSJub3RlIiBhcmlhLWxpdmU9InBvbGl0ZSI+PC9kaXY+CiAgICAgIDwvZm9ybT4KICAgIDwvZGl2PgogIDwvc2VjdGlvbj4KCiAgPHNlY3Rpb24gc3R5bGU9ImRpc3BsYXk6Z3JpZDtnYXA6MTJweCI+CiAgICA8aW5wdXQgaWQ9InNlYXJjaCIgdHlwZT0ic2VhcmNoIiBwbGFjZWhvbGRlcj0i8J+UjSBDaGVyY2hlciBkZXMgbW90cy1jbMOpcyBvdSBkZXMgbW9tZW50c+KApiI+CiAgICA8ZGl2IGNsYXNzPSJyb3ciIHN0eWxlPSJhbGlnbi1pdGVtczpjZW50ZXIiPjxoMiBpZD0iY291bnQiPkNsaXBzPC9oMj4KICAgICAgPGEgY2xhc3M9ImJ0biBnaG9zdCIgaWQ9InppcCIgc3R5bGU9ImZsZXg6MCAwIGF1dG8iIGhpZGRlbj7irIcgVG91dCB0w6lsw6ljaGFyZ2VyIChaSVApPC9hPjwvZGl2PgogICAgPGRpdiBpZD0iY2xpcHMiIHN0eWxlPSJkaXNwbGF5OmdyaWQ7Z2FwOjEycHgiPjwvZGl2PgogIDwvc2VjdGlvbj4KPC9tYWluPgoKPHNjcmlwdD4KY29uc3Qgam9iID0gbmV3IFVSTFNlYXJjaFBhcmFtcyhsb2NhdGlvbi5zZWFyY2gpLmdldCgnam9iJyk7CmNvbnN0ICQgPSBpZCA9PiBkb2N1bWVudC5nZXRFbGVtZW50QnlJZChpZCk7CmNvbnN0IERFRiA9IHsgc3Viczp0cnVlLCBmb250OidEZWphVnUgU2FucycsIHNpemU6ODQsIGNvbG9yOidGRkZGRkYnLCBoaWdobGlnaHQ6J0ZGRTUwMCcsIHBvc2l0aW9uOidiYXMnLCB3b3JkczozLCB1cHBlcjp0cnVlLAogIG91dGxpbmU6NiwgYm94OmZhbHNlLCBtdXNpY192b2x1bWU6MC4xNSwgdm9pY2Vfdm9sdW1lOjEsIHRyaW1fc3RhcnQ6MCwgdHJpbV9lbmQ6MCwgem9vbToxLCB0aXRsZTonJywgbXVzaWNfb246dHJ1ZSB9OwpsZXQgY2xpcHMgPSBbXSwgY3VycmVudCA9IG51bGwsIGRvbmUgPSBmYWxzZTsKY29uc3QgZXNjID0gcyA9PiBTdHJpbmcocykucmVwbGFjZSgvWyY8PiJdL2csIGMgPT4gKHsnJic6JyZhbXA7JywnPCc6JyZsdDsnLCc+JzonJmd0OycsJyInOicmcXVvdDsnfVtjXSkpOwoKYXN5bmMgZnVuY3Rpb24gcG9sbCgpIHsKICB0cnkgewogICAgY29uc3QgciA9IGF3YWl0IGZldGNoKCcvYXBpL2pvYnMvJyArIGpvYik7CiAgICBjb25zdCBkID0gYXdhaXQgci5qc29uKCk7CiAgICBpZiAoIXIub2spIHRocm93IG5ldyBFcnJvcihkLmRldGFpbCk7CiAgICAkKCdzdGF0dXMnKS5pbm5lckhUTUwgPSBkLnN0YXR1cyA9PT0gJ2Vycm9yJyA/ICc8c3BhbiBjbGFzcz0iZXJyIj4nICsgZXNjKGQuc3RlcCkgKyAnPC9zcGFuPicgOiBlc2MoZC5zdGVwKTsKICAgIGNvbnN0IG0gPSBkLnN0ZXAubWF0Y2goLyhcZCspXC8oXGQrKS8pOwogICAgJCgnYmFyJykuc3R5bGUud2lkdGggPSBkLnN0YXR1cyAhPT0gJ3J1bm5pbmcnID8gJzEwMCUnIDogbSA/ICg0MCArIDYwICogbVsxXSAvIG1bMl0pICsgJyUnIDogZC5zdGVwLmluY2x1ZGVzKCdUcmFuc2NyaXB0aW9uJykgPyAnMjUlJyA6ICcxMCUnOwogICAgY2xpcHMgPSBkLmNsaXBzOyBkcmF3TGlzdCgpOyByZWZyZXNoRWRpdG9yKCk7CiAgICAkKCd6aXAnKS5oaWRkZW4gPSBkLnN0YXR1cyAhPT0gJ2RvbmUnIHx8ICFjbGlwcy5sZW5ndGg7ICQoJ3ppcCcpLmhyZWYgPSBgL2FwaS9qb2JzLyR7am9ifS96aXBgOwogICAgZG9uZSA9IGQuc3RhdHVzICE9PSAncnVubmluZyc7CiAgICBjb25zdCBidXN5ID0gY2xpcHMuc29tZShjID0+IGMuYnVzeSk7CiAgICBpZiAoIWRvbmUgfHwgYnVzeSkgc2V0VGltZW91dChwb2xsLCAyNTAwKTsKICB9IGNhdGNoIChlKSB7ICQoJ3N0YXR1cycpLmlubmVySFRNTCA9ICc8c3BhbiBjbGFzcz0iZXJyIj4nICsgZXNjKGUubWVzc2FnZSkgKyAnPC9zcGFuPic7IH0KfQoKbGV0IGRyYXduID0gJyc7CmNvbnN0IEdSID0geyBob29rOiAnQWNjcm9jaGUnLCBmbG93OiAnRmx1aWRpdMOpJywgdmFsdWU6ICdWYWxldXInLCB0cmVuZDogJ1RlbmRhbmNlJyB9Owpjb25zdCBtbSA9IHQgPT4gTWF0aC5mbG9vcih0IC8gNjApLnRvU3RyaW5nKCkucGFkU3RhcnQoMiwgJzAnKSArICc6JyArIE1hdGguZmxvb3IodCAlIDYwKS50b1N0cmluZygpLnBhZFN0YXJ0KDIsICcwJyk7CmZ1bmN0aW9uIGhsKHRleHQsIHEpIHsgY29uc3QgdCA9IGVzYyh0ZXh0KTsgcmV0dXJuIHEgPyB0LnJlcGxhY2UobmV3IFJlZ0V4cCgnKCcgKyBxLnJlcGxhY2UoL1suKis/XiR7fSgpfFtcXVxcXS9nLCAnXFwkJicpICsgJyknLCAnZ2knKSwgJzxtYXJrPiQxPC9tYXJrPicpIDogdDsgfQpmdW5jdGlvbiBkcmF3TGlzdCgpIHsKICBjb25zdCBxID0gJCgnc2VhcmNoJykudmFsdWUudHJpbSgpOwogIGNvbnN0IHNob3duID0gY2xpcHMuZmlsdGVyKGMgPT4gIXEgfHwgKGMudGl0bGUgKyAnICcgKyBjLnRleHQpLnRvTG93ZXJDYXNlKCkuaW5jbHVkZXMocS50b0xvd2VyQ2FzZSgpKSk7CiAgY29uc3Qga2V5ID0gcSArICd8JyArIGN1cnJlbnQgKyAnfCcgKyBjbGlwcy5tYXAoYyA9PiBjLmZpbGUgKyBjLmJ1c3kpLmpvaW4oKTsKICBpZiAoa2V5ID09PSBkcmF3bikgcmV0dXJuOyBkcmF3biA9IGtleTsKICAkKCdjb3VudCcpLnRleHRDb250ZW50ID0gYENsaXBzICgke3Nob3duLmxlbmd0aH0ke3EgPyAnIC8gJyArIGNsaXBzLmxlbmd0aCA6ICcnfSlgOwogICQoJ2NsaXBzJykuaW5uZXJIVE1MID0gc2hvd24ubWFwKChjLCBpKSA9PiBgCiAgICA8ZGl2IGNsYXNzPSJjYXJkIHJlc3VsdCAke2N1cnJlbnQgPT09IGMubiA/ICdzZWwnIDogJyd9IiBkYXRhLW49IiR7Yy5ufSIgdGFiaW5kZXg9IjAiIHJvbGU9ImJ1dHRvbiIgYXJpYS1sYWJlbD0iTW9udGVyIGxlIGNsaXAgJHtjLm59Ij4KICAgICAgPGgzPiMke2NsaXBzLmluZGV4T2YoYykgKyAxfSAke2hsKGMudGl0bGUgfHwgJ0NsaXAgJyArIGMubiwgcSl9PC9oMz4KICAgICAgPHZpZGVvIHNyYz0iJHtjLmZpbGUuc3BsaXQoJz8nKVswXX0jdD0wLjUiIG11dGVkIHBsYXlzaW5saW5lIHByZWxvYWQ9Im1ldGFkYXRhIj48L3ZpZGVvPgogICAgICA8ZGl2PgogICAgICAgIDxkaXYgY2xhc3M9ImJpZyI+JHtjLnNjb3JlfTxzbWFsbD4vMTAwPC9zbWFsbD48L2Rpdj4KICAgICAgICA8ZGl2IGNsYXNzPSJncmFkZXMiPiR7T2JqZWN0LmVudHJpZXMoR1IpLm1hcCgoW2ssIGxdKSA9PiBgPGI+JHtjLmdyYWRlcz8uW2tdIHx8ICfigJMnfTwvYj48c3Bhbj4ke2x9PC9zcGFuPmApLmpvaW4oJycpfTwvZGl2PgogICAgICAgIDxzbWFsbCBzdHlsZT0ibWFyZ2luLXRvcDo4cHgiPiR7bW0oYy5zdGFydCl9IOKGkiAke21tKGMuZW5kKX0ke2MuYnVzeSA/ICcgwrcg4o+zIG1vbnRhZ2XigKYnIDogJyd9PC9zbWFsbD4KICAgICAgPC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9InNjZW5lIj48YiBzdHlsZT0iY29sb3I6dmFyKC0tZmcpIj5BbmFseXNlIGRlIHNjw6huZTwvYj48YnI+CiAgICAgICAgJHsoYy50cmFuc2NyaXB0IHx8IFtdKS5tYXAoeCA9PiBgPHNwYW4+WyR7bW0oeC50KX1dPC9zcGFuPiAke2hsKHgudGV4dCwgcSl9YCkuam9pbignPGJyPicpfTwvZGl2PgogICAgPC9kaXY+YCkuam9pbignJykgfHwgJzxwIGNsYXNzPSJub3RlIj5BdWN1biBjbGlwIG5lIGNvbnRpZW50IGNlcyBtb3RzLjwvcD4nOwp9CiQoJ3NlYXJjaCcpLmFkZEV2ZW50TGlzdGVuZXIoJ2lucHV0JywgZHJhd0xpc3QpOwokKCdjbGlwcycpLmFkZEV2ZW50TGlzdGVuZXIoJ2tleWRvd24nLCBlID0+IHsKICBjb25zdCBlbCA9IGUudGFyZ2V0LmNsb3Nlc3QoJy5yZXN1bHQnKTsgaWYgKGVsICYmIChlLmtleSA9PT0gJ0VudGVyJyB8fCBlLmtleSA9PT0gJyAnKSkgeyBlLnByZXZlbnREZWZhdWx0KCk7IG9wZW5FZGl0b3IoK2VsLmRhdGFzZXQubik7IH0KfSk7CmNvbnN0IFBSRVNFVFMgPSB7CiAgamF1bmU6IHsgZm9udDonTW9udHNlcnJhdCcsIHNpemU6OTYsIGNvbG9yOidGRkZGRkYnLCBoaWdobGlnaHQ6J0ZGRTUwMCcsIG91dGxpbmU6OCwgYm94OmZhbHNlLCB1cHBlcjp0cnVlLCB3b3JkczozIH0sCiAgYm9pdGU6IHsgZm9udDonRGVqYVZ1IFNhbnMnLCBzaXplOjgwLCBjb2xvcjonRkZGRkZGJywgaGlnaGxpZ2h0OicyRkQ2RUEnLCBvdXRsaW5lOjAsIGJveDp0cnVlLCB1cHBlcjpmYWxzZSwgd29yZHM6NCB9LAogIG5lb246ICB7IGZvbnQ6J1BvcHBpbnMnLCBzaXplOjkwLCBjb2xvcjonRkZGRkZGJywgaGlnaGxpZ2h0OidGRjJCRDYnLCBvdXRsaW5lOjQsIGJveDpmYWxzZSwgdXBwZXI6dHJ1ZSwgd29yZHM6MiB9LAogIG1pbmltYWw6IHsgZm9udDonRGVqYVZ1IFNhbnMnLCBzaXplOjY0LCBjb2xvcjonRkZGRkZGJywgaGlnaGxpZ2h0OidGRkZGRkYnLCBvdXRsaW5lOjMsIGJveDpmYWxzZSwgdXBwZXI6ZmFsc2UsIHdvcmRzOjYgfSwKfTsKJCgncHJlc2V0cycpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywgZSA9PiB7CiAgY29uc3QgYiA9IGUudGFyZ2V0LmNsb3Nlc3QoJ2J1dHRvbicpOyBpZiAoIWIpIHJldHVybjsKICBmb3IgKGNvbnN0IFtrLCB2XSBvZiBPYmplY3QuZW50cmllcyhQUkVTRVRTW2IuZGF0YXNldC5wXSkpIHsgY29uc3QgZWwgPSAkKCdzXycgKyBrKTsgaWYgKCFlbCkgY29udGludWU7CiAgICBpZiAoZWwudHlwZSA9PT0gJ2NoZWNrYm94JykgZWwuY2hlY2tlZCA9IHY7IGVsc2UgaWYgKGVsLnR5cGUgPT09ICdjb2xvcicpIGVsLnZhbHVlID0gJyMnICsgdi50b0xvd2VyQ2FzZSgpOyBlbHNlIGVsLnZhbHVlID0gdjsgfQogIHNob3dWYWx1ZXMoKTsgZGlydHkgPSB0cnVlOwp9KTsKbGV0IGRpcnR5ID0gZmFsc2U7CiQoJ2VkRm9ybScpLmFkZEV2ZW50TGlzdGVuZXIoJ2lucHV0JywgKCkgPT4geyBkaXJ0eSA9IHRydWU7IH0pOwokKCdjbGlwcycpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywgZSA9PiB7CiAgY29uc3QgZWwgPSBlLnRhcmdldC5jbG9zZXN0KCcucmVzdWx0Jyk7IGlmICghZWwpIHJldHVybjsKICBvcGVuRWRpdG9yKCtlbC5kYXRhc2V0Lm4pOwp9KTsKCmZ1bmN0aW9uIG9wZW5FZGl0b3IobikgewogIGlmIChkaXJ0eSAmJiBjdXJyZW50ICYmIG4gIT09IGN1cnJlbnQgJiYgIWNvbmZpcm0oJ1R1IGFzIGRlcyByw6lnbGFnZXMgbm9uIGFwcGxpcXXDqXMgc3VyIGNlIGNsaXAuIExlcyBhYmFuZG9ubmVyID8nKSkgcmV0dXJuOwogIGRpcnR5ID0gZmFsc2U7CiAgY3VycmVudCA9IG47IGRyYXduID0gJyc7IGRyYXdMaXN0KCk7CiAgY29uc3QgYyA9IGNsaXBzW24gLSAxXSwgc3QgPSB7IC4uLkRFRiwgLi4uYy5zdHlsZSB9OwogICQoJ2VkaXRvckNhcmQnKS5oaWRkZW4gPSBmYWxzZTsKICAkKCdlZFRpdGxlJykudGV4dENvbnRlbnQgPSAnTW9udGFnZSBkdSBjbGlwICcgKyBuOwogIGZvciAoY29uc3QgayBpbiBERUYpIHsKICAgIGNvbnN0IGVsID0gJCgnc18nICsgayk7IGlmICghZWwpIGNvbnRpbnVlOwogICAgaWYgKGVsLnR5cGUgPT09ICdjaGVja2JveCcpIGVsLmNoZWNrZWQgPSAhIXN0W2tdOwogICAgZWxzZSBpZiAoZWwudHlwZSA9PT0gJ2NvbG9yJykgZWwudmFsdWUgPSAnIycgKyBTdHJpbmcoc3Rba10pLnRvTG93ZXJDYXNlKCk7CiAgICBlbHNlIGVsLnZhbHVlID0gc3Rba107CiAgfQogICQoJ211c2ljJykudmFsdWUgPSAnJzsgc2hvd1ZhbHVlcygpOyBzZXRWaWRlbyhjKTsKICAkKCdlZGl0b3JDYXJkJykuc2Nyb2xsSW50b1ZpZXcoeyBiZWhhdmlvcjogJ3Ntb290aCcgfSk7Cn0KZnVuY3Rpb24gc2V0VmlkZW8oYykgewogIGlmICgkKCdlZFZpZGVvJykuZ2V0QXR0cmlidXRlKCdzcmMnKSAhPT0gYy5maWxlKSAkKCdlZFZpZGVvJykuc3JjID0gYy5maWxlOwogICQoJ2VkRG93bmxvYWQnKS5ocmVmID0gYy5maWxlLnNwbGl0KCc/JylbMF07CiAgJCgnZWRJbmZvJykudGV4dENvbnRlbnQgPSAnwqsgJyArIGMudGV4dC5zbGljZSgwLCAxNjApICsgJ+KApiDCuyc7Cn0KZnVuY3Rpb24gcmVmcmVzaEVkaXRvcigpIHsKICBpZiAoIWN1cnJlbnQpIHJldHVybjsKICBjb25zdCBjID0gY2xpcHNbY3VycmVudCAtIDFdOwogIHNldFZpZGVvKGMpOwogICQoJ2FwcGx5JykuZGlzYWJsZWQgPSBjLmJ1c3k7CiAgJCgnZWRTdGF0dXMnKS5pbm5lckhUTUwgPSBjLmJ1c3kgPyAn4o+zIE1vbnRhZ2UgZW4gY291cnPigKYnIDogYy5lcnJvciA/ICc8c3BhbiBjbGFzcz0iZXJyIj4nICsgZXNjKGMuZXJyb3IpICsgJzwvc3Bhbj4nIDogJyc7Cn0KZnVuY3Rpb24gcmVhZFN0eWxlKCkgewogIGNvbnN0IHN0ID0ge307CiAgZm9yIChjb25zdCBrIGluIERFRikgewogICAgY29uc3QgZWwgPSAkKCdzXycgKyBrKTsgaWYgKCFlbCkgY29udGludWU7CiAgICBzdFtrXSA9IGVsLnR5cGUgPT09ICdjaGVja2JveCcgPyBlbC5jaGVja2VkIDogZWwudHlwZSA9PT0gJ2NvbG9yJyA/IGVsLnZhbHVlLnNsaWNlKDEpLnRvVXBwZXJDYXNlKCkKICAgICAgOiAoZWwudHlwZSA9PT0gJ3JhbmdlJyB8fCBlbC50eXBlID09PSAnbnVtYmVyJykgPyArZWwudmFsdWUgOiBlbC52YWx1ZTsKICB9CiAgc3Qud29yZHMgPSArc3Qud29yZHM7IHJldHVybiBzdDsKfQpmdW5jdGlvbiBzaG93VmFsdWVzKCkgewogICQoJ3Zfc2l6ZScpLnRleHRDb250ZW50ID0gJCgnc19zaXplJykudmFsdWU7ICQoJ3Zfb3V0bGluZScpLnRleHRDb250ZW50ID0gJCgnc19vdXRsaW5lJykudmFsdWU7CiAgJCgndl9tdXNpY192b2x1bWUnKS50ZXh0Q29udGVudCA9IE1hdGgucm91bmQoJCgnc19tdXNpY192b2x1bWUnKS52YWx1ZSAqIDEwMCkgKyAnJSc7CiAgJCgndl92b2ljZV92b2x1bWUnKS50ZXh0Q29udGVudCA9IE1hdGgucm91bmQoJCgnc192b2ljZV92b2x1bWUnKS52YWx1ZSAqIDEwMCkgKyAnJSc7CiAgJCgndl96b29tJykudGV4dENvbnRlbnQgPSAnw5cnICsgKCskKCdzX3pvb20nKS52YWx1ZSkudG9GaXhlZCgyKTsKfQokKCdlZEZvcm0nKS5hZGRFdmVudExpc3RlbmVyKCdpbnB1dCcsIHNob3dWYWx1ZXMpOwokKCd0YWJzJykuYWRkRXZlbnRMaXN0ZW5lcignY2xpY2snLCBlID0+IHsKICBjb25zdCBiID0gZS50YXJnZXQuY2xvc2VzdCgnYnV0dG9uJyk7IGlmICghYikgcmV0dXJuOwogIGRvY3VtZW50LnF1ZXJ5U2VsZWN0b3JBbGwoJyN0YWJzIGJ1dHRvbicpLmZvckVhY2goeCA9PiB7IHguY2xhc3NMaXN0LnRvZ2dsZSgnb24nLCB4ID09PSBiKTsgeC5zZXRBdHRyaWJ1dGUoJ2FyaWEtc2VsZWN0ZWQnLCB4ID09PSBiKTsgfSk7CiAgZG9jdW1lbnQucXVlcnlTZWxlY3RvckFsbCgnW2RhdGEtcGFuZWxdJykuZm9yRWFjaChwID0+IHAuaGlkZGVuID0gcC5kYXRhc2V0LnBhbmVsICE9PSBiLmRhdGFzZXQudGFiKTsKfSk7Cgphc3luYyBmdW5jdGlvbiBzZW5kKG4sIHN0eWxlLCB3aXRoTXVzaWMpIHsKICBjb25zdCBmZCA9IG5ldyBGb3JtRGF0YSgpOwogIGZkLnNldCgnc3R5bGUnLCBKU09OLnN0cmluZ2lmeShzdHlsZSkpOwogIGlmICh3aXRoTXVzaWMgJiYgJCgnbXVzaWMnKS5maWxlc1swXSkgZmQuc2V0KCdtdXNpYycsICQoJ211c2ljJykuZmlsZXNbMF0pOwogIGNvbnN0IHIgPSBhd2FpdCBmZXRjaChgL2FwaS9qb2JzLyR7am9ifS9jbGlwcy8ke259YCwgeyBtZXRob2Q6ICdQT1NUJywgYm9keTogZmQgfSk7CiAgaWYgKCFyLm9rKSB0aHJvdyBuZXcgRXJyb3IoKGF3YWl0IHIuanNvbigpKS5kZXRhaWwpOwp9CiQoJ2VkRm9ybScpLmFkZEV2ZW50TGlzdGVuZXIoJ3N1Ym1pdCcsIGFzeW5jIGUgPT4gewogIGUucHJldmVudERlZmF1bHQoKTsKICBjb25zdCBhID0gKyQoJ3NfdHJpbV9zdGFydCcpLnZhbHVlLCBiID0gKyQoJ3NfdHJpbV9lbmQnKS52YWx1ZSwgYzAgPSBjbGlwc1tjdXJyZW50IC0gMV07CiAgaWYgKGEgKyBiID49IChjMC5lbmQgLSBjMC5zdGFydCkgLSAyKSB7ICQoJ2VkU3RhdHVzJykuaW5uZXJIVE1MID0gJzxzcGFuIGNsYXNzPSJlcnIiPkxhIGNvdXBlIGVzdCBwbHVzIGxvbmd1ZSBxdWUgbGUgY2xpcC4gUsOpZHVpcyBEw6lidXQgb3UgRmluLjwvc3Bhbj4nOyByZXR1cm47IH0KICBkaXJ0eSA9IGZhbHNlOwogIHRyeSB7IGF3YWl0IHNlbmQoY3VycmVudCwgcmVhZFN0eWxlKCksIHRydWUpOyBjbGlwc1tjdXJyZW50IC0gMV0uYnVzeSA9IHRydWU7IHJlZnJlc2hFZGl0b3IoKTsgc2V0VGltZW91dChwb2xsLCAxNTAwKTsgfQogIGNhdGNoIChlcnIpIHsgJCgnZWRTdGF0dXMnKS5pbm5lckhUTUwgPSAnPHNwYW4gY2xhc3M9ImVyciI+JyArIGVzYyhlcnIubWVzc2FnZSkgKyAnPC9zcGFuPic7IH0KfSk7CiQoJ2FwcGx5QWxsJykuYWRkRXZlbnRMaXN0ZW5lcignY2xpY2snLCBhc3luYyAoKSA9PiB7CiAgY29uc3Qgc3QgPSByZWFkU3R5bGUoKTsgZGVsZXRlIHN0LnRyaW1fc3RhcnQ7IGRlbGV0ZSBzdC50cmltX2VuZDsgZGVsZXRlIHN0Lnpvb207IGRlbGV0ZSBzdC50aXRsZTsgZGVsZXRlIHN0Lm11c2ljX29uOwogIGZvciAoY29uc3QgYyBvZiBjbGlwcykgaWYgKCFjLmJ1c3kpIGF3YWl0IHNlbmQoYy5uLCBzdCwgZmFsc2UpLmNhdGNoKCgpID0+IHt9KTsKICAkKCdlZFN0YXR1cycpLnRleHRDb250ZW50ID0gJ+KPsyBTdHlsZSBhcHBsaXF1w6kgw6AgdG91cyBsZXMgY2xpcHMsIG1vbnRhZ2UgZW4gY291cnPigKYnOwogIHNldFRpbWVvdXQocG9sbCwgMTUwMCk7Cn0pOwoKaWYgKCFqb2IpICQoJ3N0YXR1cycpLmlubmVySFRNTCA9ICc8c3BhbiBjbGFzcz0iZXJyIj5BdWN1bmUgdmlkw6lvLiA8YSBocmVmPSIvIj5Db21tZW5jZSBpY2k8L2E+Ljwvc3Bhbj4nOyBlbHNlIHBvbGwoKTsKPC9zY3JpcHQ+CjwvYm9keT4KPC9odG1sPgo=", "static/style.css": "W2hpZGRlbl0geyBkaXNwbGF5Om5vbmUgIWltcG9ydGFudDsgfQo6cm9vdCB7IC0tYmc6IzBjMTAxNTsgLS1jYXJkOiMxNTFiMjI7IC0tZmc6I2VlZjNmNzsgLS1tdXRlZDojYTNiMGJjOyAtLWxpbmU6IzI2MzAzYTsgLS1hY2NlbnQ6IzJmZDZlYTsgLS1vbi1hY2NlbnQ6IzAwMjYyYzsgLS1iYWQ6I2ZmN2E3YTsgY29sb3Itc2NoZW1lOmRhcms7IH0KKiB7IGJveC1zaXppbmc6Ym9yZGVyLWJveDsgfQpib2R5IHsgbWFyZ2luOjA7IGJhY2tncm91bmQ6dmFyKC0tYmcpOyBjb2xvcjp2YXIoLS1mZyk7IGZvbnQ6MTZweC8xLjUgc3lzdGVtLXVpLCBzYW5zLXNlcmlmOyB9Cm1haW4geyBtYXgtd2lkdGg6OTYwcHg7IG1hcmdpbjphdXRvOyBwYWRkaW5nOjIwcHggMTZweCA2MHB4OyBkaXNwbGF5OmdyaWQ7IGdhcDoxNnB4OyB9CmgxIHsgbWFyZ2luOjA7IGZvbnQtc2l6ZTozMHB4OyB9IGgxIHNwYW4geyBjb2xvcjp2YXIoLS1hY2NlbnQpOyB9IGgxIGEgeyBjb2xvcjppbmhlcml0OyB0ZXh0LWRlY29yYXRpb246bm9uZTsgfQpoMiB7IG1hcmdpbjowOyBmb250LXNpemU6MjBweDsgfQouc3ViLCAubm90ZSB7IGNvbG9yOnZhcigtLW11dGVkKTsgbWFyZ2luOjA7IH0gLm5vdGUgeyBmb250LXNpemU6MTNweDsgfQouY2FyZCB7IGJhY2tncm91bmQ6dmFyKC0tY2FyZCk7IGJvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7IGJvcmRlci1yYWRpdXM6MTRweDsgcGFkZGluZzoxNnB4OyBkaXNwbGF5OmdyaWQ7IGdhcDoxMnB4OyBtaW4td2lkdGg6MDsgfQpsYWJlbCB7IGRpc3BsYXk6Z3JpZDsgZ2FwOjRweDsgZm9udC1zaXplOjE0cHg7IGNvbG9yOnZhcigtLW11dGVkKTsgfQpsYWJlbC5jaGVjayB7IGRpc3BsYXk6ZmxleDsgYWxpZ24taXRlbXM6Y2VudGVyOyBnYXA6OHB4OyBjb2xvcjp2YXIoLS1mZyk7IGZvbnQtc2l6ZToxNXB4OyB9CmxhYmVsLmNoZWNrIGlucHV0IHsgd2lkdGg6MjJweDsgaGVpZ2h0OjIycHg7IH0KaW5wdXQsIHNlbGVjdCB7IHdpZHRoOjEwMCU7IHBhZGRpbmc6MTFweDsgYm9yZGVyLXJhZGl1czoxMHB4OyBib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpOyBiYWNrZ3JvdW5kOiMwZjEyMTk7IGNvbG9yOnZhcigtLWZnKTsgZm9udC1zaXplOjE2cHg7IH0KaW5wdXRbdHlwZT1jb2xvcl0geyBwYWRkaW5nOjJweDsgaGVpZ2h0OjQ0cHg7IH0gaW5wdXRbdHlwZT1yYW5nZV0geyBwYWRkaW5nOjA7IH0KLmdyaWQgeyBkaXNwbGF5OmdyaWQ7IGdyaWQtdGVtcGxhdGUtY29sdW1uczpyZXBlYXQoYXV0by1maXQsIG1pbm1heCgxNDBweCwgMWZyKSk7IGdhcDoxMHB4OyB9CmJ1dHRvbiwgLmJ0biB7IHBhZGRpbmc6MTNweCAxNnB4OyBib3JkZXI6MDsgYm9yZGVyLXJhZGl1czoxMnB4OyBiYWNrZ3JvdW5kOnZhcigtLWFjY2VudCk7IGNvbG9yOnZhcigtLW9uLWFjY2VudCk7IGZvbnQtd2VpZ2h0OjcwMDsgZm9udC1zaXplOjE2cHg7IGN1cnNvcjpwb2ludGVyOyB0ZXh0LWFsaWduOmNlbnRlcjsgdGV4dC1kZWNvcmF0aW9uOm5vbmU7IH0KYnV0dG9uLmdob3N0LCAuYnRuLmdob3N0IHsgYmFja2dyb3VuZDp0cmFuc3BhcmVudDsgY29sb3I6dmFyKC0tZmcpOyBib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpOyB9CmJ1dHRvbjpkaXNhYmxlZCB7IG9wYWNpdHk6LjU7IGN1cnNvcjp3YWl0OyB9Ci5vciB7IHRleHQtYWxpZ246Y2VudGVyOyBjb2xvcjp2YXIoLS1tdXRlZCk7IGZvbnQtc2l6ZToxM3B4OyB0ZXh0LXRyYW5zZm9ybTp1cHBlcmNhc2U7IGxldHRlci1zcGFjaW5nOi4wOGVtOyB9CiNzdGF0dXMgeyBjb2xvcjp2YXIoLS1hY2NlbnQpOyBmb250LXdlaWdodDo2MDA7IH0gLmVyciB7IGNvbG9yOnZhcigtLWJhZCk7IH0KLmJhciB7IGhlaWdodDo2cHg7IGJhY2tncm91bmQ6dmFyKC0tbGluZSk7IGJvcmRlci1yYWRpdXM6M3B4OyBvdmVyZmxvdzpoaWRkZW47IH0gLmJhciBpIHsgZGlzcGxheTpibG9jazsgaGVpZ2h0OjEwMCU7IHdpZHRoOjUlOyBiYWNrZ3JvdW5kOnZhcigtLWFjY2VudCk7IHRyYW5zaXRpb246d2lkdGggLjVzOyB9Ci5jbGlwcyB7IGRpc3BsYXk6Z3JpZDsgZ3JpZC10ZW1wbGF0ZS1jb2x1bW5zOnJlcGVhdChhdXRvLWZpbGwsIG1pbm1heCgxNzBweCwgMWZyKSk7IGdhcDoxMnB4OyB9Ci5jbGlwIHsgY3Vyc29yOnBvaW50ZXI7IH0gLmNsaXAuc2VsIHsgb3V0bGluZToycHggc29saWQgdmFyKC0tYWNjZW50KTsgfQouY2xpcCB2aWRlbywgLnByZXZpZXcgdmlkZW8geyB3aWR0aDoxMDAlOyBhc3BlY3QtcmF0aW86OS8xNjsgYm9yZGVyLXJhZGl1czoxMHB4OyBiYWNrZ3JvdW5kOiMwMDA7IGRpc3BsYXk6YmxvY2s7IH0KLnNjb3JlIHsgZm9udC13ZWlnaHQ6NzAwOyBjb2xvcjp2YXIoLS1hY2NlbnQpOyB9CnNtYWxsIHsgY29sb3I6dmFyKC0tbXV0ZWQpOyBkaXNwbGF5OmJsb2NrOyB9Ci5lZGl0b3IgeyBkaXNwbGF5OmdyaWQ7IGdyaWQtdGVtcGxhdGUtY29sdW1uczptaW5tYXgoMCwgMzAwcHgpIG1pbm1heCgwLCAxZnIpOyBnYXA6MTZweDsgYWxpZ24taXRlbXM6c3RhcnQ7IH0KLnRhYnMgeyBkaXNwbGF5OmZsZXg7IGZsZXgtd3JhcDp3cmFwOyBnYXA6NnB4OyB9Ci50YWJzIGJ1dHRvbiB7IHBhZGRpbmc6OHB4IDEycHg7IGZvbnQtc2l6ZToxNHB4OyBiYWNrZ3JvdW5kOnRyYW5zcGFyZW50OyBjb2xvcjp2YXIoLS1mZyk7IGJvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7IH0KLnRhYnMgYnV0dG9uLm9uIHsgYmFja2dyb3VuZDp2YXIoLS1hY2NlbnQpOyBjb2xvcjp2YXIoLS1vbi1hY2NlbnQpOyBib3JkZXItY29sb3I6dmFyKC0tYWNjZW50KTsgfQoucm93IHsgZGlzcGxheTpmbGV4OyBnYXA6MTBweDsgZmxleC13cmFwOndyYXA7IH0gLnJvdyA+ICogeyBmbGV4OjE7IH0KQG1lZGlhIChtYXgtd2lkdGg6NjQwcHgpIHsgLmVkaXRvciB7IGdyaWQtdGVtcGxhdGUtY29sdW1uczoxZnI7IH0gLnByZXZpZXcgeyBtYXgtd2lkdGg6MjYwcHg7IG1hcmdpbjphdXRvOyB9IH0KLmhlcm8geyB0ZXh0LWFsaWduOmNlbnRlcjsgZGlzcGxheTpncmlkOyBnYXA6MTBweDsgcGFkZGluZzoyOHB4IDAgOHB4OyB9Ci5oZXJvIGgyIHsgZm9udC1zaXplOmNsYW1wKDI2cHgsIDZ2dywgNDJweCk7IGxpbmUtaGVpZ2h0OjEuMTsgfQouZHJvcCB7IGRpc3BsYXk6ZmxleDsgZ2FwOjhweDsgYmFja2dyb3VuZDp2YXIoLS1jYXJkKTsgYm9yZGVyOjFweCBzb2xpZCB2YXIoLS1saW5lKTsgYm9yZGVyLXJhZGl1czo5OTlweDsgcGFkZGluZzo2cHg7IH0KLmRyb3AgaW5wdXQgeyBib3JkZXI6MDsgYmFja2dyb3VuZDp0cmFuc3BhcmVudDsgYm9yZGVyLXJhZGl1czo5OTlweDsgfQouZHJvcCBidXR0b24geyBib3JkZXItcmFkaXVzOjk5OXB4OyB3aGl0ZS1zcGFjZTpub3dyYXA7IH0KLnRvb2xzIHsgZGlzcGxheTpncmlkOyBncmlkLXRlbXBsYXRlLWNvbHVtbnM6cmVwZWF0KGF1dG8tZml0LCBtaW5tYXgoMTIwcHgsIDFmcikpOyBnYXA6MTBweDsgfQoudG9vbCB7IGJhY2tncm91bmQ6dmFyKC0tY2FyZCk7IGJvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7IGJvcmRlci1yYWRpdXM6MTJweDsgcGFkZGluZzoxMnB4OyB0ZXh0LWFsaWduOmNlbnRlcjsgZm9udC1zaXplOjE0cHg7IH0KLnRvb2wgYiB7IGRpc3BsYXk6YmxvY2s7IGZvbnQtc2l6ZToyMnB4OyB9Ci5zZWcgeyBkaXNwbGF5OmlubGluZS1mbGV4OyBib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpOyBib3JkZXItcmFkaXVzOjk5OXB4OyBwYWRkaW5nOjNweDsgZ2FwOjNweDsgfQouc2VnIGJ1dHRvbiB7IHBhZGRpbmc6N3B4IDE0cHg7IGJvcmRlci1yYWRpdXM6OTk5cHg7IGJhY2tncm91bmQ6dHJhbnNwYXJlbnQ7IGNvbG9yOnZhcigtLWZnKTsgZm9udC1zaXplOjE0cHg7IH0KLnNlZyBidXR0b24ub24geyBiYWNrZ3JvdW5kOnZhcigtLWFjY2VudCk7IGNvbG9yOnZhcigtLW9uLWFjY2VudCk7IH0KLmR1YWwgeyBwb3NpdGlvbjpyZWxhdGl2ZTsgaGVpZ2h0OjMwcHg7IH0KLmR1YWwgaW5wdXQgeyBwb3NpdGlvbjphYnNvbHV0ZTsgaW5zZXQ6MDsgcG9pbnRlci1ldmVudHM6bm9uZTsgYmFja2dyb3VuZDpub25lOyBib3JkZXI6MDsgcGFkZGluZzowOyAtd2Via2l0LWFwcGVhcmFuY2U6bm9uZTsgYXBwZWFyYW5jZTpub25lOyB9Ci5kdWFsIGlucHV0Ojotd2Via2l0LXNsaWRlci10aHVtYiB7IHBvaW50ZXItZXZlbnRzOmF1dG87IC13ZWJraXQtYXBwZWFyYW5jZTpub25lOyB3aWR0aDoxOHB4OyBoZWlnaHQ6MThweDsgYm9yZGVyLXJhZGl1czo1MCU7IGJhY2tncm91bmQ6dmFyKC0tYWNjZW50KTsgY3Vyc29yOnBvaW50ZXI7IH0KLmR1YWwgaW5wdXQ6Oi1tb3otcmFuZ2UtdGh1bWIgeyBwb2ludGVyLWV2ZW50czphdXRvOyB3aWR0aDoxOHB4OyBoZWlnaHQ6MThweDsgYm9yZGVyLXJhZGl1czo1MCU7IGJhY2tncm91bmQ6dmFyKC0tYWNjZW50KTsgYm9yZGVyOjA7IH0KLmR1YWwgLnRyYWNrIHsgcG9zaXRpb246YWJzb2x1dGU7IGxlZnQ6MDsgcmlnaHQ6MDsgdG9wOjEzcHg7IGhlaWdodDo0cHg7IGJhY2tncm91bmQ6dmFyKC0tbGluZSk7IGJvcmRlci1yYWRpdXM6MnB4OyB9Ci5kdWFsIC5maWxsIHsgcG9zaXRpb246YWJzb2x1dGU7IHRvcDoxM3B4OyBoZWlnaHQ6NHB4OyBiYWNrZ3JvdW5kOnZhcigtLWFjY2VudCk7IGJvcmRlci1yYWRpdXM6MnB4OyB9Ci50aW1lcyB7IGRpc3BsYXk6ZmxleDsganVzdGlmeS1jb250ZW50OnNwYWNlLWJldHdlZW47IGZvbnQtdmFyaWFudC1udW1lcmljOnRhYnVsYXItbnVtczsgZm9udC1zaXplOjEzcHg7IGNvbG9yOnZhcigtLW11dGVkKTsgfQoudGh1bWIgeyB3aWR0aDoxMDAlOyBhc3BlY3QtcmF0aW86MTYvOTsgb2JqZWN0LWZpdDpjb3ZlcjsgYm9yZGVyLXJhZGl1czoxMHB4OyBiYWNrZ3JvdW5kOiMwMDA7IGRpc3BsYXk6YmxvY2s7IH0KLnByb2plY3RzIHsgZGlzcGxheTpncmlkOyBncmlkLXRlbXBsYXRlLWNvbHVtbnM6cmVwZWF0KGF1dG8tZmlsbCwgbWlubWF4KDE2MHB4LCAxZnIpKTsgZ2FwOjEycHg7IH0KLnByb2plY3QgeyB0ZXh0LWRlY29yYXRpb246bm9uZTsgY29sb3I6dmFyKC0tZmcpOyB9Ci5wcm9qZWN0IHZpZGVvIHsgd2lkdGg6MTAwJTsgYXNwZWN0LXJhdGlvOjkvMTY7IG9iamVjdC1maXQ6Y292ZXI7IGJvcmRlci1yYWRpdXM6MTBweDsgYmFja2dyb3VuZDojMDAwOyB9Ci5yZXN1bHQgeyBkaXNwbGF5OmdyaWQ7IGdyaWQtdGVtcGxhdGUtY29sdW1uczoxNTBweCAxMTBweCBtaW5tYXgoMCwgMWZyKTsgZ2FwOjE0cHg7IGFsaWduLWl0ZW1zOnN0YXJ0OyBjdXJzb3I6cG9pbnRlcjsgfQoucmVzdWx0LnNlbCB7IG91dGxpbmU6MnB4IHNvbGlkIHZhcigtLWFjY2VudCk7IH0KLnJlc3VsdCB2aWRlbyB7IHdpZHRoOjEwMCU7IGFzcGVjdC1yYXRpbzo5LzE2OyBib3JkZXItcmFkaXVzOjEwcHg7IGJhY2tncm91bmQ6IzAwMDsgfQoucmVzdWx0IGgzIHsgbWFyZ2luOjAgMCA4cHg7IGZvbnQtc2l6ZToxOHB4OyBncmlkLWNvbHVtbjoxLy0xOyB9Ci5iaWcgeyBmb250LXNpemU6MzBweDsgZm9udC13ZWlnaHQ6ODAwOyBjb2xvcjp2YXIoLS1hY2NlbnQpOyBsaW5lLWhlaWdodDoxOyB9IC5iaWcgc21hbGwgeyBkaXNwbGF5OmlubGluZTsgZm9udC1zaXplOjE0cHg7IH0KLmdyYWRlcyB7IGRpc3BsYXk6Z3JpZDsgZ3JpZC10ZW1wbGF0ZS1jb2x1bW5zOmF1dG8gMWZyOyBnYXA6MnB4IDhweDsgZm9udC1zaXplOjE0cHg7IG1hcmdpbi10b3A6OHB4OyB9Ci5ncmFkZXMgYiB7IGNvbG9yOnZhcigtLWFjY2VudCk7IH0KLnNjZW5lIHsgZm9udC1zaXplOjE0cHg7IG1heC1oZWlnaHQ6MjQwcHg7IG92ZXJmbG93OmF1dG87IGNvbG9yOnZhcigtLW11dGVkKTsgfQouc2NlbmUgc3BhbiB7IGNvbG9yOnZhcigtLWFjY2VudCk7IGZvbnQtdmFyaWFudC1udW1lcmljOnRhYnVsYXItbnVtczsgfQptYXJrIHsgYmFja2dyb3VuZDp2YXIoLS1hY2NlbnQpOyBjb2xvcjp2YXIoLS1vbi1hY2NlbnQpOyBib3JkZXItcmFkaXVzOjNweDsgcGFkZGluZzowIDJweDsgfQpAbWVkaWEgKG1heC13aWR0aDo2NDBweCkgeyAucmVzdWx0IHsgZ3JpZC10ZW1wbGF0ZS1jb2x1bW5zOjExMHB4IG1pbm1heCgwLCAxZnIpOyB9IC5yZXN1bHQgLnNjZW5lIHsgZ3JpZC1jb2x1bW46MS8tMTsgfSB9CmJ1dHRvbjpmb2N1cy12aXNpYmxlLCAuYnRuOmZvY3VzLXZpc2libGUsIC5yZXN1bHQ6Zm9jdXMtdmlzaWJsZSwgaW5wdXQ6Zm9jdXMtdmlzaWJsZSwgc2VsZWN0OmZvY3VzLXZpc2libGUgeyBvdXRsaW5lOjJweCBzb2xpZCB2YXIoLS1hY2NlbnQpOyBvdXRsaW5lLW9mZnNldDoycHg7IH0KLnByZXNldHMgeyBkaXNwbGF5OmZsZXg7IGZsZXgtd3JhcDp3cmFwOyBnYXA6OHB4OyBncmlkLWNvbHVtbjoxLy0xOyB9Ci5wcmVzZXRzIGJ1dHRvbiB7IHBhZGRpbmc6OHB4IDEycHg7IGZvbnQtc2l6ZToxNHB4OyBiYWNrZ3JvdW5kOnRyYW5zcGFyZW50OyBjb2xvcjp2YXIoLS1mZyk7IGJvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7IH0K"}
app = pathlib.Path("/content/clipnova")
for name, data in FILES.items():
    f = app / name
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_bytes(base64.b64decode(data))
print("✅ ClipNova copié dans", app)

In [ ]:
# 3) Démarrage + lien à ouvrir sur ton iPad
import re, subprocess, time
subprocess.run("pkill -f 'uvicorn server:app'; pkill -f cloudflared", shell=True)
server = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd="/content/clipnova", stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
import urllib.request
for _ in range(60):  # attend que l'application réponde avant de donner le lien
    try:
        urllib.request.urlopen("http://localhost:8000/", timeout=2)
        break
    except Exception:
        time.sleep(1)
if url:
    from IPython.display import HTML, display
    display(HTML(f'<h2>👉 <a href="{url}" target="_blank">{url}</a></h2>'
                 '<p>Ouvre ce lien sur ton iPad. Garde cet onglet Colab ouvert.</p>'))
else:
    print("❌ Le lien n'a pas pu être créé. Relance cette cellule.")

## Bon à savoir
- **Première vidéo plus lente** : le modèle de sous-titres se télécharge une fois (≈ 500 Mo pour « Normale »).
- **Vidéos de plus de 100 Mo** : le lien ne les accepte pas. Raccourcis la vidéo avant, ou choisis une qualité plus basse.
- **Liens YouTube** : YouTube bloque souvent les serveurs de Google. Si le lien ne marche pas, importe le fichier.
- **Darija / arabe** : choisis la langue « Arabe / Darija » et la précision « Haute ».
- En cas de problème, regarde le journal : exécute `!tail -n 30 /content/server.log`.